# Chapter 5: Visualizing Medical Data with ggplot2

## 1. Introduction

Welcome to the next step in your data visualization journey. In the previous section, you learned the basic grammar of `ggplot2`. Now, we will apply that knowledge to create three fundamental plot types that are essential for medical data analysis: scatter, box, and violin plots. These visualizations allow us to uncover relationships between variables, compare distributions across patient groups, and gain initial insights from complex datasets. By the end of this section, you will be able to choose the right plot for your question, create it using `ggplot2`, and interpret its clinical significance.

---

## 2. Key Concepts and Definitions

*   **Scatter Plot**: A graph that uses Cartesian coordinates to display values for two continuous variables. In medicine, it's used to visualize the relationship or correlation between two measurements, such as age and blood pressure, to identify trends or associations.
*   **Box Plot (or Box-and-Whisker Plot)**: A standardized way of displaying the distribution of data based on a five-number summary (minimum, first quartile (Q1), median, third quartile (Q3), and maximum). It is excellent for comparing the central tendency and spread of a variable across different groups, like comparing the efficacy of a new treatment against a control.
*   **Violin Plot**: A method of plotting numeric data that combines a box plot with a kernel density plot. It shows the full probability density of the data, revealing more about the distribution shape (e.g., whether it's bimodal or skewed) than a simple box plot. This is useful for visualizing distributions of biomarkers, like white blood cell counts, to see where data points are concentrated.
*   **Jitter**: The technique of adding a small amount of random noise to the location of data points on a plot. This is used to prevent overplotting when many data points have the same value, making it easier to see the underlying distribution, especially with small sample sizes.

---

## 3. Main Content

This section provides the core code and explanations for creating fundamental plot types. We will first set up a sample patient dataset and then walk through each plot.

### 3.1 Setup: Patient Dataset

First, we load the necessary libraries and create a sample `tibble` containing patient data. This dataset simulates common medical measurements and groupings, incorporating agent feedback for concise variable names and improved clinical realism.

```R
# Load the entire Tidyverse suite for a consistent workflow
library(tidyverse)

patient_data <- tibble(
  age = c(45, 62, 33, 51, 28, 68, 41, 55, 39, 29, 58, 49),
  sbp = c(128, 155, 120, 145, 118, 160, 125, 148, 130, 115, 150, 142), # Systolic BP (mmHg)
  treatment_group = factor(rep(c("Control", "Treatment"), each = 6)),
  glucose = c(95, 88, 102, 91, 85, 98, 82, 75, 79, 88, 70, 72), # Glucose (mg/dL)
  patient_status = factor(c("Healthy", "Infection", "Healthy", "Infection", "Healthy", "Infection", "Healthy", "Infection", "Healthy", "Infection", "Infection", "Healthy")),
  wbc_count = c(7.5, 12.1, 6.8, 14.3, 8.0, 15.1, 6.5, 13.5, 7.1, 11.8, 14.8, 8.2) # WBC (10^9/L)
)
```

### 3.2 Scatter Plots

Use a scatter plot to visualize the relationship between two continuous variables, like patient age and systolic blood pressure.

```R
ggplot(patient_data, aes(x = age, y = sbp, color = patient_status)) +
  geom_point(size = 3) +
  labs(
    title = "Patient Age vs. Systolic Blood Pressure",
    x = "Age (Years)",
    y = "Systolic BP (mmHg)",
    color = "Patient Status"
  ) +
  theme_bw()
```

**Interpretation:** The plot shows patients with an 'Infection' tend to be older with higher *systolic* blood pressure, suggesting a potential positive correlation.

> **Medical Background:** Hypertension (high blood pressure) is a major risk factor for cardiovascular disease, and its prevalence increases significantly with age. Visualizing this relationship helps clinicians and researchers identify at-risk patient populations and understand disease progression.

### 3.3 Box Plots

While scatter plots are excellent for continuous variables, we need a different tool to compare a continuous measurement across categorical groups. This is where box plots excel. They are used to compare summary statistics (median, quartiles) of a measurement across different groups, such as glucose levels in treatment versus control.

```R
ggplot(patient_data, aes(x = treatment_group, y = glucose)) +
  geom_boxplot() +
  labs(
    title = "Glucose Levels by Treatment Group",
    x = "Treatment Group",
    y = "Glucose (mg/dL)"
  ) +
  theme_bw()
```

**Interpretation:** The plot confirms the median glucose level (the horizontal line inside the box) is visibly lower in the 'Treatment' group compared to the 'Control' group.

> **Key Terms:** A box plot visualizes the "five-number summary" of a dataset: the minimum, first quartile (25th percentile), median (50th percentile), third quartile (75th percentile), and maximum. The "box" itself represents the interquartile range (IQR), which contains the middle 50% of the data.

> **In Practice:** Comparing treatment and control groups is the foundation of clinical trials. Box plots provide a rapid, clear visual summary to assess if a new therapy (like a glucose-lowering drug) is having a measurable effect compared to a placebo or standard care.

### 3.4 Violin Plots

A box plot summarizes the data, but what if we want to see the full distribution shape? For that, we turn to the violin plot. It shows the full probability density of data, revealing the distribution shape for groups, like WBC counts in healthy versus infected patients.

```R
ggplot(patient_data, aes(x = patient_status, y = wbc_count)) +
  geom_violin() +
  labs(
    title = "WBC Distribution by Patient Status",
    x = "Patient Status",
    y = "WBC Count (10^9/L)"
  ) +
  theme_bw()
```

**Interpretation:** The 'Infection' group's violin is wider at higher WBC counts, showing a concentration of patients with elevated levels (leukocytosis).

> **Medical Background:** Leukocytosis, an elevated white blood cell (WBC) count, is a common indicator of the body's response to infection, inflammation, or certain diseases like leukemia. A violin plot effectively shows how the distribution of WBCs in the 'Infection' group is shifted towards these higher, clinically significant values.

### 3.5 Combined Plots for Richer Summary

Overlay a box plot on a violin plot to combine a statistical summary (median) with the full data distribution (density).

```R
ggplot(patient_data, aes(x = patient_status, y = wbc_count)) +
  geom_violin(alpha = 0.6) +
  geom_boxplot(width = 0.1, fill = "white",
               outlier.shape = NA) + # Hides boxplot outliers, as the violin already shows the full data density.
  labs(
    title = "WBC Distribution by Patient Status",
    x = "Patient Status",
    y = "WBC Count (10^9/L)"
  ) +
  theme_bw()
```

**Interpretation:** This combined plot confirms the 'Infection' group has both a higher median WBC count (from the box) and a distribution skewed towards higher values (from the violin).

> **Pro Tip:** Layering `geoms` (like `geom_violin` and `geom_boxplot`) is a powerful feature of `ggplot2`. It allows you to build complex, information-rich visualizations step-by-step. This combination gives you the best of both worlds: a view of the distribution's shape and a precise summary of its central tendency.

### 3.6 Adding Raw Data with Jitter

For small datasets, add jittered points over a summary plot to show the raw data, revealing the true sample size and distribution.

```R
set.seed(42) # Ensures jitter is reproducible for consistent teaching materials
ggplot(patient_data, aes(x = treatment_group, y = glucose)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.5) +
  geom_jitter(width = 0.2, alpha = 0.7) + # Overlays raw data points with slight horizontal noise.
  labs(
    title = "Glucose Levels by Treatment Group with Raw Data",
    x = "Treatment Group",
    y = "Glucose (mg/dL)"
  ) +
  theme_bw()
```

**Interpretation:** `geom_jitter()` reveals all individual data points, confirming the sample size and variability within each group that the boxplot alone summarizes.

> **Important:** With small datasets, summary plots like box plots can be misleading. They might hide the true distribution or small sample size. Adding `geom_jitter()` is crucial as it displays every individual data point, providing a more honest and complete picture of the data.

---

---

## 4. Practice Exercises

### Exercise 1: Scatter Plot for Inflammation and Blood Pressure

**Objective:** Create and interpret a scatter plot to identify potential correlations between two continuous variables.
**Time:** 5 minutes
**Medical Context:** Investigating if systemic inflammation (indicated by White Blood Cell count) is associated with hypertension (high systolic blood pressure).

**Task:** Write the code to create a scatter plot of `wbc_count` (x-axis) vs. `sbp` (y-axis). Then, state whether the visual shows a positive, negative, or no clear correlation.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
ggplot(patient_data, aes(x = wbc_count, y = sbp)) +
  geom_point(size = 3) +
  labs(
    title = "WBC Count vs. Systolic Blood Pressure",
    x = "WBC Count (10^9/L)",
    y = "Systolic BP (mmHg)"
  ) +
  theme_bw()
```

**Explanation:** The plot shows a clear positive correlation. As the White Blood Cell (WBC) count increases, the systolic blood pressure also tends to increase. This suggests a potential link between inflammation and hypertension in this dataset.
**Key Learning:** Scatter plots are the primary tool for visualizing the relationship and direction of correlation between two continuous variables.

</div>
</details>

### Exercise 2: Combined Plot for Glucose Status

**Objective:** Combine plot types to compare both summary statistics and the full distribution shape between groups.
**Time:** 7 minutes
**Medical Context:** Comparing glucose regulation between healthy and infected patient groups to understand metabolic differences that may be associated with the disease state.

**Task:** Write the code for a combined violin and boxplot to compare `glucose` across `patient_status`. Identify which group has a lower median and which shows a wider data distribution.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
ggplot(patient_data, aes(x = patient_status, y = glucose)) +
  geom_violin(alpha = 0.6) +
  geom_boxplot(width = 0.1, fill = "white", outlier.shape = NA) +
  labs(
    title = "Glucose Distribution by Patient Status",
    x = "Patient Status",
    y = "Glucose (mg/dL)"
  ) +
  theme_bw()
```

**Explanation:** The 'Infection' group has a lower median glucose level (the line in the box is lower). However, the 'Healthy' group's violin is wider, indicating a broader range and wider distribution of glucose values in this sample.
**Key Learning:** Combining plots provides a more nuanced view. While the median might be lower in one group, the other group might have greater variability.

</div>
</details>

### Exercise 3: Evaluating Treatment Effect on Blood Pressure

**Objective:** Use a box plot with raw data points to evaluate the effect of a treatment on a continuous variable.
**Time:** 8 minutes
**Medical Context:** Assessing the impact of a new antihypertensive medication by comparing blood pressure readings between the treatment and control groups.

**Task:** Create a box plot showing `sbp` for each `treatment_group`. Overlay jittered points to show the raw data. Interpret the result.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
set.seed(42) # For reproducible jitter
ggplot(patient_data, aes(x = treatment_group, y = sbp)) +
  geom_boxplot(alpha = 0.5, outlier.shape = NA) +
  geom_jitter(width = 0.2, alpha = 0.7, aes(color = treatment_group)) +
  labs(
    title = "Systolic BP by Treatment Group",
    x = "Treatment Group",
    y = "Systolic BP (mmHg)"
  ) +
  theme_bw() +
  theme(legend.position = "none") # Hide legend as colors are redundant
```

**Explanation:** The box plot for the 'Treatment' group is positioned higher than the 'Control' group, indicating a higher median systolic blood pressure. This is contrary to what one would expect from an effective antihypertensive treatment, suggesting the "Treatment" in this simulated dataset was not effective for BP.
**Key Learning:** Box plots with jitter are excellent for comparing groups and simultaneously verifying the underlying data, which is crucial for drawing accurate conclusions in clinical data analysis.

</div>
</details>

---

---

## 5. Practical Applications

*   **Clinical Trial Efficacy Analysis:** Box plots are frequently used to compare a primary endpoint (e.g., tumor size reduction, blood pressure change) between a treatment arm and a placebo arm. The visualization provides an immediate, clear comparison of the median effect and the variability of patient responses, which is critical for interim and final trial reports submitted to regulatory bodies like the FDA.
*   **Identifying Patient Subgroups with Genomics Data:** Violin plots are powerful for visualizing gene expression data from RNA-sequencing. For example, a researcher could plot the expression of a cancer-related gene across different tumor subtypes. A bimodal (two-humped) distribution in one violin might reveal two distinct patient subgroups within that subtype, one with high expression and one with low, suggesting different underlying biology that could guide personalized treatment strategies.
*   **Public Health and Epidemiology:** Scatter plots are essential for exploring relationships between population-level variables. An epidemiologist might plot vaccination rates against disease incidence for different regions to visually assess the effectiveness of a vaccination campaign. By adding a regression line (`geom_smooth(method = "lm")`), they can quantify this relationship and strengthen public health recommendations.

---

---

## 6. Summary and Key Takeaways

In this section, we've explored how to create and interpret three fundamental plot types in `ggplot2` for medical data analysis. You learned to move beyond basic bar charts to visualize relationships, compare distributions, and add layers of information to your plots.

*   **Scatter plots** are ideal for visualizing the relationship between two continuous variables, helping to identify potential correlations like age and blood pressure.
*   **Box plots** provide a concise summary of a variable's distribution (median, quartiles, range) and are excellent for comparing these statistics across different groups (e.g., treatment vs. control).
*   **Violin plots** show the full probability density of the data, offering a more detailed view of the distribution's shape than a box plot, which is useful for spotting multi-modal or skewed data patterns.
*   Combining plot types, such as overlaying box plots on violin plots or adding jittered raw data points, creates richer visualizations that convey more information in a single, powerful figure.

With these foundational plotting skills, you are now ready to move on to customizing the aesthetics and annotations of your plots to make them publication-ready.

> **Reflection Moment:** You have learned to create three distinct plot types. When would you choose a violin plot over a box plot for a clinical presentation? What are the advantages and disadvantages of showing the raw data with `geom_jitter` to a non-technical audience like a patient?

---